# Home Credit Default Risk — Notebook 1  
## Prepare and Aggregate External Tables for Modelling

This notebook prepares the **non-application tables** from the Kaggle Home Credit Default Risk dataset and converts them into a **customer-level feature table** keyed by `SK_ID_CURR`.

### Why this notebook exists
In the original logistic workflow, the model used application-level variables plus a small number of engineered affordability and stability features. The next improvement step is to bring in **bureau**, **previous applications**, and **behavioural repayment data** from the other Home Credit tables so the later logistic regression notebook can test whether these external signals improve discrimination.

### Output
At the end of this notebook you will save:

- `home_credit_external_features.csv`
- `home_credit_external_feature_catalog.csv`

### Expected input files
Place the Kaggle CSV files in a folder such as `../data/`:

- `application_train.csv`
- `application_test.csv`
- `bureau.csv`
- `bureau_balance.csv`
- `previous_application.csv`
- `POS_CASH_balance.csv`
- `installments_payments.csv`
- `credit_card_balance.csv`

This notebook is written to be **portfolio-friendly**:
- clear sectioning
- explicit feature logic
- reusable helper functions
- customer-level output ready for modelling


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

DATA_DIR = Path(r"D:\Jane\Job Search\credit-risk-portfolio_bank\1. Probability of Default Modelling and Score Card_Home Loan\data")

# List all files in the data directory
print(list(DATA_DIR.glob("*")))


In [ ]:
PROCESSED_DIR = Path(r"D:\Jane\Job Search\credit-risk-portfolio_bank\1. Probability of Default Modelling and Score Card_Home Loan\processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())

## 1. Helper functions

These helpers keep the notebook clean and make the feature preparation logic easier to explain in an interview or portfolio review.


In [ ]:
# Reduces RAM usage by converting data types:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to reduce memory footprint."""
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df

# Safely divide two numbers or Series, returning NaN for zero or missing denominators.

def safe_divide(numerator, denominator):
    """Divide safely and return NaN where denominator is zero or missing."""
    if isinstance(denominator, pd.Series):
        denominator = denominator.replace(0, np.nan)
    elif denominator == 0:
        denominator = np.nan
    return numerator / denominator

# Summarizes missing values and unique counts for each column in a DataFrame.
def missing_summary(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    out = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "n_unique": df.nunique(dropna=True).values
    }).sort_values(["missing_pct", "n_unique"], ascending=[False, False])
    return out.head(top_n)

# Groups a DataFrame by a key and aggregates with prefixed, flattened column names.
def group_aggregate(df: pd.DataFrame, group_key: str, prefix: str, agg_dict: dict) -> pd.DataFrame:
    """Group and aggregate with flattened, prefixed column names."""
    grouped = df.groupby(group_key).agg(agg_dict)
    grouped.columns = [f"{prefix}_{col}_{stat}".upper() for col, stat in grouped.columns]
    return grouped.reset_index()

# Prints the shape of a DataFrame with a formatted name.
def print_shape(name: str, df: pd.DataFrame) -> None:
    print(f"{name:<30} shape = {df.shape}")


## 2. Load the master customer ID list

We use both `application_train` and `application_test` so the external feature table can later be merged into either dataset.


In [ ]:
app_train = pd.read_csv(DATA_DIR / "application_train.csv", usecols=["SK_ID_CURR", "TARGET"])

app_test = pd.read_csv(DATA_DIR / "application_test.csv", usecols=["SK_ID_CURR"])

master_ids = pd.concat(
    [
        app_train[["SK_ID_CURR"]],
        app_test[["SK_ID_CURR"]]
    ],
    axis=0,
    ignore_index=True
).drop_duplicates().reset_index(drop=True)

print_shape("application_train", app_train)
print_shape("application_test", app_test)
print_shape("master_ids", master_ids)

master_ids.head()


## 3. Bureau and bureau balance features

### Modelling idea
These two tables describe the applicant's **external credit history**.

Key signals we want:
- total outside exposure
- current debt load
- overdue behaviour
- recency of bureau records
- how often bureau accounts show delinquency

Bureau Dataset — Column Definitions: 

SK_ID_CURR	            Unique customer ID linking to application data
SK_ID_BUREAU	        Unique ID for each credit account (one customer can have multiple records)
CREDIT_ACTIVE	        Status of the credit account (Active, Closed, Sold)
CREDIT_TYPE	            Type of credit product (e.g. credit card, consumer loan, mortgage)
DAYS_CREDIT	Days        since the credit was granted (negative = in the past)
DAYS_CREDIT_ENDDATE	    Expected end date of the credit (relative to application date)
DAYS_ENDDATE_FACT	    Actual end date of the credit (if closed)
AMT_CREDIT_MAX_OVERDUE	Maximum overdue amount historically recorded
AMT_CREDIT_SUM	        Total credit amount granted
AMT_CREDIT_SUM_DEBT	    Current outstanding debt amount
AMT_CREDIT_SUM_OVERDUE	Current overdue amount
AMT_ANNUITY	Regular     repayment amount (instalment)
CNT_CREDIT_PROLONG	    Number of times the credit was extended/prolonged

Bureau Balance Dataset — Column Definitions

SK_ID_BUREAU	Unique credit account ID linking to bureau dataset
MONTHS_BALANCE	Month index relative to application date (0 = current, -1 = previous month)
STATUS	Monthly repayment status (0 = no delay, 1–5 = delinquency severity, C = closed, X = no data)

In [ ]:
bureau_cols = [
    "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "CREDIT_TYPE",
    "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY", "CNT_CREDIT_PROLONG"
]

bureau_balance_cols = ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"]

bureau = pd.read_csv(DATA_DIR / "bureau.csv", usecols=bureau_cols)

bureau_balance = pd.read_csv(DATA_DIR / "bureau_balance.csv",  usecols=bureau_balance_cols)

bureau = reduce_memory_usage(bureau)
bureau_balance = reduce_memory_usage(bureau_balance)

print_shape("bureau", bureau)
print_shape("bureau_balance", bureau_balance)


The STATUS field represents the monthly repayment status of each credit account.

Original Value	Meaning	Interpretation (Credit Risk)
X	No loan / no data for the month	No activity → not informative (often treated as missing)
C	Credit closed	Loan fully repaid → generally positive signal
0	No delay (current)	Borrower is paying on time → good behaviour
1	1–30 days past due	Minor delinquency → early warning signal
2	31–60 days past due	Moderate delinquency → increasing risk
3	61–90 days past due	Serious delinquency → high risk
4	91–120 days past due	Severe delinquency → very high risk
5	120+ days past due / default	Default-level delinquency → strongest negative signal

X:Separate “no info” from real repayment behaviour
C and 0: Both indicate no delinquency risk
1–5:Preserve severity of delinquency

In [ ]:
status_map = {
    "X": -1,   # no loan for the month / no status
    "C": 0,    # closed
    "0": 0,    # current
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5
}

bureau_balance["STATUS_NUM"] = bureau_balance["STATUS"].map(status_map)
bureau_balance["HAS_DPD"] = (bureau_balance["STATUS_NUM"] > 0).astype("int8")
bureau_balance["HAS_SEVERE_DPD"] = (bureau_balance["STATUS_NUM"] >= 3).astype("int8")

bb_agg = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),
    BB_MONTHS_MAX=("MONTHS_BALANCE", "max"),
    BB_RECORD_COUNT=("MONTHS_BALANCE", "size"),
    BB_STATUS_MEAN=("STATUS_NUM", "mean"),
    BB_STATUS_MAX=("STATUS_NUM", "max"),
    BB_DPD_RATE=("HAS_DPD", "mean"),
    BB_SEVERE_DPD_RATE=("HAS_SEVERE_DPD", "mean")
).reset_index()

print_shape("bureau_balance_agg", bb_agg)
bb_agg.head()


In [ ]:
print(bureau["SK_ID_BUREAU"].dtype)
print(bb_agg["SK_ID_BUREAU"].dtype)

In [ ]:
bureau["SK_ID_BUREAU"] = bureau["SK_ID_BUREAU"].astype("int64")
bb_agg["SK_ID_BUREAU"] = bb_agg["SK_ID_BUREAU"].astype("int64")

bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

DAYS_CREDIT:Number of days since the loan was granted,Always negative (because it’s in the past),Recent borrowing → possibly higher risk, Older borrowing → more stable history

DAYS_ENDDATE_FACT:Actual date when the loan was closed (repaid), advice Repayment behaviour, Recently closed loans → good repayment track record,Many closed loans → experienced borrower

DAYS_CREDIT_ENDDATE:Expected end date of the loan (contractual),advice early repayment or delayed repayment

Below Feature Engineering Explaination:
BUREAU_DAYS_CREDIT_MEAN:Average time of past loans,Lower (more negative) → older credit history,Higher (closer to 0) → more recent borrowing

BUREAU_DAYS_CREDIT_MIN:Oldest loan ever taken,Measures length of credit history,long credit history → GOOD,thin file → RISKY

BUREAU_DAYS_CREDIT_MAX:Most recent loan,Close to 0 → recently took credit → possible stress

BUREAU_DAYS_ENDDATE_FACT_MAX:Most recently closed loan to find the latest repayment behaviour,Recently closed loan → good repayment ability,No recent closure → maybe stuck in debt

BUREAU_CNT_CREDIT_PROLONG_SUM:Total number of times borrower extended loans,High → borrower struggles to repay on time,Strong financial stress signal

Active loans:
BUREAU_IS_ACTIVE_SUM:Number of currently active loans
BUREAU_IS_ACTIVE_MEAN:% of loans that are active

BUREAU_IS_CONSUMER_CREDIT_MEAN:% of loans that are consumer loans,High consumer credit → unsecured exposure

BUREAU_IS_CREDIT_CARD_MEAN: % of loans that are credit cards,High credit card usage → revolving risk

BB_RECORD_COUNT = number of months per loan
BB_STATUS_MEAN = average delinquency
BB_STATUS_MAX = worst delinquency
BB_DPD_RATE = % months overdue
BB_SEVERE_DPD_RATE = % severe delinquency

bb_agg amd bureau are merged at the SK_ID_BUREAU level = one row per loan/account, below  model needs features at SK_ID_CURR level = one row per customer

In [ ]:

bureau["DEBT_TO_CREDIT_RATIO"] = safe_divide(bureau["AMT_CREDIT_SUM_DEBT"], bureau["AMT_CREDIT_SUM"])

bureau["OVERDUE_TO_DEBT_RATIO"] = safe_divide(bureau["AMT_CREDIT_SUM_OVERDUE"], bureau["AMT_CREDIT_SUM_DEBT"])

bureau["IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype("int8")

bureau["IS_CLOSED"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype("int8")

bureau["IS_CONSUMER_CREDIT"] = (bureau["CREDIT_TYPE"] == "Consumer credit").astype("int8")

bureau["IS_CREDIT_CARD"] = (bureau["CREDIT_TYPE"] == "Credit card").astype("int8")

bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_DAYS_CREDIT_ENDDATE_MAX=("DAYS_CREDIT_ENDDATE", "max"),
    BUREAU_DAYS_ENDDATE_FACT_MAX=("DAYS_ENDDATE_FACT", "max"),
    BUREAU_AMT_CREDIT_SUM_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_AMT_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_AMT_CREDIT_SUM_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_AMT_CREDIT_SUM_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    BUREAU_CNT_CREDIT_PROLONG_SUM=("CNT_CREDIT_PROLONG", "sum"),
    BUREAU_DEBT_TO_CREDIT_RATIO_MEAN=("DEBT_TO_CREDIT_RATIO", "mean"),
    BUREAU_DEBT_TO_CREDIT_RATIO_MAX=("DEBT_TO_CREDIT_RATIO", "max"),
    BUREAU_OVERDUE_TO_DEBT_RATIO_MEAN=("OVERDUE_TO_DEBT_RATIO", "mean"),
    BUREAU_IS_ACTIVE_SUM=("IS_ACTIVE", "sum"),
    BUREAU_IS_ACTIVE_MEAN=("IS_ACTIVE", "mean"),
    BUREAU_IS_CLOSED_SUM=("IS_CLOSED", "sum"),
    BUREAU_IS_CONSUMER_CREDIT_MEAN=("IS_CONSUMER_CREDIT", "mean"),
    BUREAU_IS_CREDIT_CARD_MEAN=("IS_CREDIT_CARD", "mean"),
    BUREAU_BB_RECORD_COUNT_SUM=("BB_RECORD_COUNT", "sum"),
    BUREAU_BB_STATUS_MEAN_MEAN=("BB_STATUS_MEAN", "mean"),
    BUREAU_BB_STATUS_MAX_MAX=("BB_STATUS_MAX", "max"),
    BUREAU_BB_DPD_RATE_MEAN=("BB_DPD_RATE", "mean"),
    BUREAU_BB_SEVERE_DPD_RATE_MEAN=("BB_SEVERE_DPD_RATE", "mean")
).reset_index()

print_shape("bureau_agg", bureau_agg)
bureau_agg.head()


## 4. Previous application features

### Modelling idea
This table captures **past application behaviour** with Home Credit.

Useful signals:
- how many previous applications the customer made
- approval vs refusal pattern
- requested amount vs granted amount
- historical term and pricing structure

Column Definitions
NAME_CONTRACT_STATUS:
Status of the previous loan application (e.g. Approved, Refused, Cancelled, Unused offer)
AMT_APPLICATION:
Amount the customer originally applied for
→ reflects requested borrowing demand
AMT_DOWN_PAYMENT:
Initial payment made by the borrower
→ indicates borrower’s own capital contribution
AMT_GOODS_PRICE:
Price of the goods financed by the loan
→ used to assess loan-to-value (LTV-like measure)
RATE_DOWN_PAYMENT:
Proportion of goods price paid upfront
→ AMT_DOWN_PAYMENT / AMT_GOODS_PRICE
→ higher = lower risk (more borrower commitment)
DAYS_DECISION:
Days since the application decision was made (negative = in the past)
→ indicates recency of past credit applications
CNT_PAYMENT:
Number of instalments planned for the loan
→ proxy for loan tenor (longer = higher risk exposure)
DAYS_TERMINATION:
Actual or expected termination date of the previous loan
→ helps assess loan completion and repayment behaviour

In [ ]:
prev_cols = [
    "SK_ID_CURR", "SK_ID_PREV", "NAME_CONTRACT_STATUS",
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT",
    "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT",
    "DAYS_DECISION", "CNT_PAYMENT", "DAYS_TERMINATION"
]

prev = pd.read_csv(DATA_DIR / "previous_application.csv", usecols=prev_cols)
prev = reduce_memory_usage(prev)
print_shape("previous_application", prev)


In [ ]:
prev["APP_CREDIT_PCT"] = safe_divide(prev["AMT_APPLICATION"], prev["AMT_CREDIT"])

prev["DOWNPAYMENT_TO_CREDIT"] = safe_divide(prev["AMT_DOWN_PAYMENT"], prev["AMT_CREDIT"])

prev["GOODS_TO_CREDIT"] = safe_divide(prev["AMT_GOODS_PRICE"], prev["AMT_CREDIT"])

prev["IS_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype("int8")

prev["IS_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")

prev["IS_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype("int8")

prev["IS_UNUSED_OFFER"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype("int8")

prev_agg = prev.groupby("SK_ID_CURR").agg(
    PREV_COUNT=("SK_ID_PREV", "count"),
    PREV_AMT_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
    PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_AMT_CREDIT_SUM=("AMT_CREDIT", "sum"),
    PREV_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_APP_CREDIT_PCT_MEAN=("APP_CREDIT_PCT", "mean"),
    PREV_APP_CREDIT_PCT_MAX=("APP_CREDIT_PCT", "max"),
    PREV_DOWNPAYMENT_TO_CREDIT_MEAN=("DOWNPAYMENT_TO_CREDIT", "mean"),
    PREV_GOODS_TO_CREDIT_MEAN=("GOODS_TO_CREDIT", "mean"),
    PREV_RATE_DOWN_PAYMENT_MEAN=("RATE_DOWN_PAYMENT", "mean"),
    PREV_CNT_PAYMENT_MEAN=("CNT_PAYMENT", "mean"),
    PREV_CNT_PAYMENT_MAX=("CNT_PAYMENT", "max"),
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    PREV_IS_APPROVED_SUM=("IS_APPROVED", "sum"),
    PREV_IS_APPROVED_MEAN=("IS_APPROVED", "mean"),
    PREV_IS_REFUSED_SUM=("IS_REFUSED", "sum"),
    PREV_IS_REFUSED_MEAN=("IS_REFUSED", "mean"),
    PREV_IS_CANCELED_MEAN=("IS_CANCELED", "mean"),
    PREV_IS_UNUSED_OFFER_MEAN=("IS_UNUSED_OFFER", "mean")
).reset_index()

print_shape("prev_agg", prev_agg)
prev_agg.head()


## 5. POS cash balance features

### Modelling idea
This is useful behavioural data for instalment-based products.

Useful signals:
- number of prior POS cash records
- delinquency measures (`SK_DPD`, `SK_DPD_DEF`)
- whether historical contracts tended to complete cleanly


In [ ]:
pos_cols = [
    "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE", "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE", "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"
]

pos = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv", usecols=pos_cols)
pos = reduce_memory_usage(pos)
print_shape("POS_CASH_balance", pos)


In [ ]:
pos["IS_COMPLETED"] = (pos["NAME_CONTRACT_STATUS"] == "Completed").astype("int8")
pos["IS_ACTIVE"] = (pos["NAME_CONTRACT_STATUS"] == "Active").astype("int8")
pos["IS_SIGNED"] = (pos["NAME_CONTRACT_STATUS"] == "Signed").astype("int8")

pos_agg = pos.groupby("SK_ID_CURR").agg(
    POS_COUNT=("SK_ID_PREV", "count"),
    POS_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    POS_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    POS_CNT_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
    POS_CNT_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_SK_DPD_MEAN=("SK_DPD", "mean"),
    POS_SK_DPD_MAX=("SK_DPD", "max"),
    POS_SK_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
    POS_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    POS_IS_COMPLETED_MEAN=("IS_COMPLETED", "mean"),
    POS_IS_ACTIVE_MEAN=("IS_ACTIVE", "mean"),
    POS_IS_SIGNED_MEAN=("IS_SIGNED", "mean")
).reset_index()

print_shape("pos_agg", pos_agg)
pos_agg.head()


## 6. Installments payment features

### Modelling idea
This table is especially important because it contains **repayment discipline**.

Useful signals:
- payment delays
- underpayment frequency
- instalment shortfall
- average payment ratio


In [ ]:
inst_cols = [
    "SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"
]

inst = pd.read_csv(DATA_DIR / "installments_payments.csv", usecols=inst_cols)
inst = reduce_memory_usage(inst)
print_shape("installments_payments", inst)


In [ ]:
inst["PAYMENT_RATIO"] = safe_divide(inst["AMT_PAYMENT"], inst["AMT_INSTALMENT"])
inst["PAYMENT_SHORTFALL"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]

inst["DPD"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).clip(lower=0)
inst["DBD"] = (inst["DAYS_INSTALMENT"] - inst["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

inst["IS_LATE"] = (inst["DPD"] > 0).astype("int8")
inst["IS_UNDERPAID"] = (inst["PAYMENT_SHORTFALL"] > 0).astype("int8")

inst_agg = inst.groupby("SK_ID_CURR").agg(
    INSTAL_COUNT=("SK_ID_PREV", "count"),
    INSTAL_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
    INSTAL_PAYMENT_RATIO_MIN=("PAYMENT_RATIO", "min"),
    INSTAL_PAYMENT_SHORTFALL_SUM=("PAYMENT_SHORTFALL", "sum"),
    INSTAL_DPD_MEAN=("DPD", "mean"),
    INSTAL_DPD_MAX=("DPD", "max"),
    INSTAL_DBD_MEAN=("DBD", "mean"),
    INSTAL_IS_LATE_MEAN=("IS_LATE", "mean"),
    INSTAL_IS_UNDERPAID_MEAN=("IS_UNDERPAID", "mean"),
    INSTAL_AMT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    INSTAL_AMT_INSTALMENT_MEAN=("AMT_INSTALMENT", "mean")
).reset_index()

print_shape("inst_agg", inst_agg)
inst_agg.head()


## 7. Credit card balance features

### Modelling idea
This table helps capture revolving credit behaviour.

Useful signals:
- card utilization
- current drawings relative to limit
- payment strength relative to required minimum
- card delinquency


In [ ]:
cc_cols = [
    "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE",
    "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT",
    "SK_DPD", "SK_DPD_DEF", "CNT_DRAWINGS_CURRENT"
]

cc = pd.read_csv(DATA_DIR / "credit_card_balance.csv", usecols=cc_cols)
cc = reduce_memory_usage(cc)
print_shape("credit_card_balance", cc)


In [ ]:
cc["UTILIZATION"] = safe_divide(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
cc["DRAWINGS_TO_LIMIT_RATIO"] = safe_divide(cc["AMT_DRAWINGS_CURRENT"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
cc["PAYMENT_TO_MIN_RATIO"] = safe_divide(cc["AMT_PAYMENT_CURRENT"], cc["AMT_INST_MIN_REGULARITY"])
cc["IS_DELAYED"] = (cc["SK_DPD"] > 0).astype("int8")

cc_agg = cc.groupby("SK_ID_CURR").agg(
    CC_COUNT=("SK_ID_PREV", "count"),
    CC_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    CC_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    CC_AMT_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_AMT_BALANCE_SUM=("AMT_BALANCE", "sum"),
    CC_AMT_CREDIT_LIMIT_ACTUAL_MAX=("AMT_CREDIT_LIMIT_ACTUAL", "max"),
    CC_UTILIZATION_MEAN=("UTILIZATION", "mean"),
    CC_UTILIZATION_MAX=("UTILIZATION", "max"),
    CC_DRAWINGS_TO_LIMIT_RATIO_MEAN=("DRAWINGS_TO_LIMIT_RATIO", "mean"),
    CC_PAYMENT_TO_MIN_RATIO_MEAN=("PAYMENT_TO_MIN_RATIO", "mean"),
    CC_SK_DPD_MEAN=("SK_DPD", "mean"),
    CC_SK_DPD_MAX=("SK_DPD", "max"),
    CC_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    CC_IS_DELAYED_MEAN=("IS_DELAYED", "mean"),
    CC_CNT_DRAWINGS_CURRENT_MEAN=("CNT_DRAWINGS_CURRENT", "mean")
).reset_index()

print_shape("cc_agg", cc_agg)
cc_agg.head()


## 8. Merge all external feature tables

At this point each source table has been collapsed to one row per customer.  
Now we combine them into one modelling-ready external feature table.


In [ ]:
external_features = (
    master_ids
    .merge(bureau_agg, on="SK_ID_CURR", how="left")
    .merge(prev_agg, on="SK_ID_CURR", how="left")
    .merge(pos_agg, on="SK_ID_CURR", how="left")
    .merge(inst_agg, on="SK_ID_CURR", how="left")
    .merge(cc_agg, on="SK_ID_CURR", how="left")
)

print_shape("external_features", external_features)
external_features.head()


In [ ]:
missing_summary(external_features, top_n=25)


In [ ]:
source_counts = pd.DataFrame({
    "source_table": ["bureau", "previous_application", "POS_CASH_balance", "installments_payments", "credit_card_balance"],
    "n_features_created": [
        bureau_agg.shape[1] - 1,
        prev_agg.shape[1] - 1,
        pos_agg.shape[1] - 1,
        inst_agg.shape[1] - 1,
        cc_agg.shape[1] - 1
    ]
})

source_counts


## 9. Save the outputs

These files are what the second notebook will use for the modelling workflow.


In [ ]:
external_feature_path = PROCESSED_DIR / "home_credit_external_features.csv"
feature_catalog_path = PROCESSED_DIR / "home_credit_external_feature_catalog.csv"

external_features.to_csv(external_feature_path, index=False)

feature_catalog = pd.DataFrame({
    "feature_name": [c for c in external_features.columns if c != "SK_ID_CURR"],
    "source_family": [
        "bureau" if c.startswith("BUREAU_")
        else "previous_application" if c.startswith("PREV_")
        else "pos_cash" if c.startswith("POS_")
        else "installments" if c.startswith("INSTAL_")
        else "credit_card" if c.startswith("CC_")
        else "other"
        for c in external_features.columns if c != "SK_ID_CURR"
    ]
})

feature_catalog.to_csv(feature_catalog_path, index=False)

print("Saved:", external_feature_path)
print("Saved:", feature_catalog_path)
print(feature_catalog.groupby("source_family").size().sort_values(ascending=False))


## 10. Portfolio wrap-up

### What this notebook achieved
- read the external Home Credit source tables
- engineered intuitive credit-risk features
- aggregated all history to the customer level
- exported a clean feature table for modelling

### Why this matters
This step moves the project from a simple application-only scorecard-style model toward a broader credit-risk view that includes:
- external credit exposure
- prior approval behaviour
- repayment discipline
- revolving credit usage

The next notebook will merge these new features back into `application_train.csv`, repeat the exploration and feature-selection flow, and rerun logistic regression to check whether ROC AUC improves.
